## Exoplanet Identification

Given *data collected about objects in space*, let's try to predict whether a given object is an **exoplanet** or not.

We will use a variety of classification models to make our predictions.

Data source: https://www.kaggle.com/datasets/nasa/kepler-exoplanet-search-results

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings(action='ignore')

In [2]:
data = pd.read_csv('archive/cumulative.csv')

In [3]:
data

,rowid,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_period_err1,koi_period_err2,koi_time0bk,koi_time0bk_err1,koi_time0bk_err2,koi_impact,koi_impact_err1,koi_impact_err2,koi_duration,koi_duration_err1,koi_duration_err2,koi_depth,koi_depth_err1,koi_depth_err2,koi_prad,koi_prad_err1,koi_prad_err2,koi_teq,koi_teq_err1,koi_teq_err2,koi_insol,koi_insol_err1,koi_insol_err2,koi_model_snr,koi_tce_plnt_num,koi_tce_delivname,koi_steff,koi_steff_err1,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag
0,1,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,9.488036,2.775000e-05,-2.775000e-05,170.538750,0.002160,-0.002160,0.146,0.318,-0.146,2.95750,0.08190,-0.08190,615.8,19.5,-19.5,2.26,0.26,-0.15,793.0,NaN,NaN,93.59,29.45,-16.65,35.8,1.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
1,2,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,0,0,0,0,54.418383,2.479000e-04,-2.479000e-04,162.513840,0.003520,-0.003520,0.586,0.059,-0.443,4.50700,0.11600,-0.11600,874.8,35.5,-35.5,2.83,0.32,-0.19,443.0,NaN,NaN,9.11,2.87,-1.62,25.8,2.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
2,3,10811496,K00753.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,0,19.899140,1.494000e-05,-1.494000e-05,175.850252,0.000581,-0.000581,0.969,5.126,-0.077,1.78220,0.03410,-0.03410,10829.0,171.0,-171.0,14.60,3.92,-1.31,638.0,NaN,NaN,39.30,31.04,-10.49,76.3,1.0,q1_q17_dr25_tce,5853.0,158.0,-176.0,4.544,0.044,-0.176,0.868,0.233,-0.078,297.00482,48.134129,15.436
3,4,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,0,1.736952,2.630000e-07,-2.630000e-07,170.307565,0.000115,-0.000115,1.276,0.115,-0.092,2.40641,0.00537,-0.00537,8079.2,12.8,-12.8,33.46,8.50,-2.83,1395.0,NaN,NaN,891.96,668.95,-230.35,505.6,1.0,q1_q17_dr25_tce,5805.0,157.0,-174.0,4.564,0.053,-0.168,0.791,0.201,-0.067,285.53461,48.285210,15.597
4,5,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,2.525592,3.761000e-06,-3.761000e-06,171.595550,0.001130,-0.001130,0.701,0.235,-0.478,1.65450,0.04200,-0.04200,603.3,16.9,-16.9,2.75,0.88,-0.35,1406.0,NaN,NaN,926.16,874.33,-314.24,40.9,1.0,q1_q17_dr25_tce,6031.0,169.0,-211.0,4.438,0.070,-0.210,1.046,0.334,-0.133,288.75488,48.226200,15.509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9559,9560,10031643,K07984.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,0,0,1,8.589871,1.846000e-04,-1.846000e-04,132.016100,0.015700,-0.015700,0.765,0.023,-0.541,4.80600,0.63400,-0.63400,87.7,13.0,-13.0,1.11,0.32,-0.23,929.0,NaN,NaN,176.40,152.77,-77.60,8.4,1.0,q1_q17_dr25_tce,5638.0,169.0,-152.0,4.296,0.231,-0.189,1.088,0.313,-0.228,298.74921,46.973351,14.478
9560,9561,10090151,K07985.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,1,0,0.527699,1.160000e-07,-1.160000e-07,131.705093,0.000170,-0.000170,1.252,0.051,-0.049,3.22210,0.01740,-0.01740,1579.2,4.6,-4.6,29.35,7.70,-2.57,2088.0,NaN,NaN,4500.53,3406.38,-1175.26,453.3,1.0,q1_q17_dr25_tce,5638.0,139.0,-166.0,4.529,0.035,-0.196,0.903,0.237,-0.079,297.18875,47.093819,14.082
9561,9562,10128825,K07986.01,NaN,CANDIDATE,CANDIDATE,0.497,0,0,0,0,1.739849,1.780000e-05,-1.780000e-05,133.001270,0.007690,-0.007690,0.043,0.423,-0.043,3.11400,0.22900,-0.22900,48.5,5.4,-5.4,0.72,0.24,-0.08,1608.0,NaN,NaN,1585.81,1537.86,-502.22,10.6,1.0,q1_q17_dr25_tce,6119.0,165.0,-220.0,4.444,0.056,-0.224,1.031,0.341,-0.114,286.50937,47.163219,14.757
9562,9563,10147276,K07987.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.021,0,0,1,0,0.681402,2.434000e-06,-2.434000e-06,132.181750,0.002850,-0.002850,0.147,0.309,-0.147,0.86500,0.16200,-0.16200,103.6,14.7,-14.7,1.07,0.36,-0.11,2218.0,NaN,NaN,5713.41,5675.74,-1836.94,12

In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 9564 entries, 0 to 9563
Data columns (total 50 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rowid              9564 non-null   int64  
 1   kepid              9564 non-null   int64  
 2   kepoi_name         9564 non-null   str    
 3   kepler_name        2294 non-null   str    
 4   koi_disposition    9564 non-null   str    
 5   koi_pdisposition   9564 non-null   str    
 6   koi_score          8054 non-null   float64
 7   koi_fpflag_nt      9564 non-null   int64  
 8   koi_fpflag_ss      9564 non-null   int64  
 9   koi_fpflag_co      9564 non-null   int64  
 10  koi_fpflag_ec      9564 non-null   int64  
 11  koi_period         9564 non-null   float64
 12  koi_period_err1    9110 non-null   float64
 13  koi_period_err2    9110 non-null   float64
 14  koi_time0bk        9564 non-null   float64
 15  koi_time0bk_err1   9110 non-null   float64
 16  koi_time0bk_err2   9110 non-null   

### Preprocessing

In [5]:
df = data.copy()

In [6]:
# Drop unused columns
df = df.drop(['rowid', 'kepid', 'kepoi_name', 'kepler_name', 'koi_pdisposition', 'koi_score'], axis=1)

In [7]:
df

,koi_disposition,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_period_err1,koi_period_err2,koi_time0bk,koi_time0bk_err1,koi_time0bk_err2,koi_impact,koi_impact_err1,koi_impact_err2,koi_duration,koi_duration_err1,koi_duration_err2,koi_depth,koi_depth_err1,koi_depth_err2,koi_prad,koi_prad_err1,koi_prad_err2,koi_teq,koi_teq_err1,koi_teq_err2,koi_insol,koi_insol_err1,koi_insol_err2,koi_model_snr,koi_tce_plnt_num,koi_tce_delivname,koi_steff,koi_steff_err1,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag
0,CONFIRMED,0,0,0,0,9.488036,2.775000e-05,-2.775000e-05,170.538750,0.002160,-0.002160,0.146,0.318,-0.146,2.95750,0.08190,-0.08190,615.8,19.5,-19.5,2.26,0.26,-0.15,793.0,NaN,NaN,93.59,29.45,-16.65,35.8,1.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
1,CONFIRMED,0,0,0,0,54.418383,2.479000e-04,-2.479000e-04,162.513840,0.003520,-0.003520,0.586,0.059,-0.443,4.50700,0.11600,-0.11600,874.8,35.5,-35.5,2.83,0.32,-0.19,443.0,NaN,NaN,9.11,2.87,-1.62,25.8,2.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
2,FALSE POSITIVE,0,1,0,0,19.899140,1.494000e-05,-1.494000e-05,175.850252,0.000581,-0.000581,0.969,5.126,-0.077,1.78220,0.03410,-0.03410,10829.0,171.0,-171.0,14.60,3.92,-1.31,638.0,NaN,NaN,39.30,31.04,-10.49,76.3,1.0,q1_q17_dr25_tce,5853.0,158.0,-176.0,4.544,0.044,-0.176,0.868,0.233,-0.078,297.00482,48.134129,15.436
3,FALSE POSITIVE,0,1,0,0,1.736952,2.630000e-07,-2.630000e-07,170.307565,0.000115,-0.000115,1.276,0.115,-0.092,2.40641,0.00537,-0.00537,8079.2,12.8,-12.8,33.46,8.50,-2.83,1395.0,NaN,NaN,891.96,668.95,-230.35,505.6,1.0,q1_q17_dr25_tce,5805.0,157.0,-174.0,4.564,0.053,-0.168,0.791,0.201,-0.067,285.53461,48.285210,15.597
4,CONFIRMED,0,0,0,0,2.525592,3.761000e-06,-3.761000e-06,171.595550,0.001130,-0.001130,0.701,0.235,-0.478,1.65450,0.04200,-0.04200,603.3,16.9,-16.9,2.75,0.88,-0.35,1406.0,NaN,NaN,926.16,874.33,-314.24,40.9,1.0,q1_q17_dr25_tce,6031.0,169.0,-211.0,4.438,0.070,-0.210,1.046,0.334,-0.133,288.75488,48.226200,15.509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9559,FALSE POSITIVE,0,0,0,1,8.589871,1.846000e-04,-1.846000e-04,132.016100,0.015700,-0.015700,0.765,0.023,-0.541,4.80600,0.63400,-0.63400,87.7,13.0,-13.0,1.11,0.32,-0.23,929.0,NaN,NaN,176.40,152.77,-77.60,8.4,1.0,q1_q17_dr25_tce,5638.0,169.0,-152.0,4.296,0.231,-0.189,1.088,0.313,-0.228,298.74921,46.973351,14.478
9560,FALSE POSITIVE,0,1,1,0,0.527699,1.160000e-07,-1.160000e-07,131.705093,0.000170,-0.000170,1.252,0.051,-0.049,3.22210,0.01740,-0.01740,1579.2,4.6,-4.6,29.35,7.70,-2.57,2088.0,NaN,NaN,4500.53,3406.38,-1175.26,453.3,1.0,q1_q17_dr25_tce,5638.0,139.0,-166.0,4.529,0.035,-0.196,0.903,0.237,-0.079,297.18875,47.093819,14.082
9561,CANDIDATE,0,0,0,0,1.739849,1.780000e-05,-1.780000e-05,133.001270,0.007690,-0.007690,0.043,0.423,-0.043,3.11400,0.22900,-0.22900,48.5,5.4,-5.4,0.72,0.24,-0.08,1608.0,NaN,NaN,1585.81,1537.86,-502.22,10.6,1.0,q1_q17_dr25_tce,6119.0,165.0,-220.0,4.444,0.056,-0.224,1.031,0.341,-0.114,286.50937,47.163219,14.757
9562,FALSE POSITIVE,0,0,1,0,0.681402,2.434000e-06,-2.434000e-06,132.181750,0.002850,-0.002850,0.147,0.309,-0.147,0.86500,0.16200,-0.16200,103.6,14.7,-14.7,1.07,0.36,-0.11,2218.0,NaN,NaN,5713.41,5675.74,-1836.94,12.3,1.0,q1_q17_dr25_tce,6173.0,193.0,-236.0,4.447,0.056,-0.224,1.041,0.341,-0.114,294.16489,47.176281,15.385


In [8]:
df['koi_disposition'].value_counts()

koi_disposition
FALSE POSITIVE    5023
CONFIRMED         2293
CANDIDATE         2248
Name: count, dtype: int64

In [9]:
# Limit target values to CANDIDATE and CONFIRMED
false_positive_rows = df.query('koi_disposition == "FALSE POSITIVE"').index
df = df.drop(false_positive_rows, axis=0).reset_index(drop=True)

In [10]:
df['koi_disposition'].value_counts()

koi_disposition
CONFIRMED    2293
CANDIDATE    2248
Name: count, dtype: int64

In [11]:
df.isna().mean()

koi_disposition      0.000000
koi_fpflag_nt        0.000000
koi_fpflag_ss        0.000000
koi_fpflag_co        0.000000
koi_fpflag_ec        0.000000
koi_period           0.000000
koi_period_err1      0.017177
koi_period_err2      0.017177
koi_time0bk          0.000000
koi_time0bk_err1     0.017177
koi_time0bk_err2     0.017177
koi_impact           0.014094
koi_impact_err1      0.017177
koi_impact_err2      0.017177
koi_duration         0.000000
koi_duration_err1    0.017177
koi_duration_err2    0.017177
koi_depth            0.014094
koi_depth_err1       0.017177
koi_depth_err2       0.017177
koi_prad             0.014094
koi_prad_err1        0.014094
koi_prad_err2        0.014094
koi_teq              0.014094
koi_teq_err1         1.000000
koi_teq_err2         1.000000
koi_insol            0.013874
koi_insol_err1       0.013874
koi_insol_err2       0.013874
koi_model_snr        0.014094
koi_tce_plnt_num     0.016516
koi_tce_delivname    0.016516
koi_steff            0.014094
koi_steff_

In [12]:
# Drop columns with all missing values
df = df.drop(['koi_teq_err1', 'koi_teq_err2'], axis=1)

In [13]:
(df.isna().mean() >= 0.25).sum()

np.int64(0)

In [14]:
df.isna().sum()

koi_disposition       0
koi_fpflag_nt         0
koi_fpflag_ss         0
koi_fpflag_co         0
koi_fpflag_ec         0
koi_period            0
koi_period_err1      78
koi_period_err2      78
koi_time0bk           0
koi_time0bk_err1     78
koi_time0bk_err2     78
koi_impact           64
koi_impact_err1      78
koi_impact_err2      78
koi_duration          0
koi_duration_err1    78
koi_duration_err2    78
koi_depth            64
koi_depth_err1       78
koi_depth_err2       78
koi_prad             64
koi_prad_err1        64
koi_prad_err2        64
koi_teq              64
koi_insol            63
koi_insol_err1       63
koi_insol_err2       63
koi_model_snr        64
koi_tce_plnt_num     75
koi_tce_delivname    75
koi_steff            64
koi_steff_err1       72
koi_steff_err2       85
koi_slogg            64
koi_slogg_err1       72
koi_slogg_err2       72
koi_srad             64
koi_srad_err1        72
koi_srad_err2        72
ra                    0
dec                   0
koi_kepmag      

In [15]:
# Fill remaining missing values
df['koi_tce_delivname'] = df['koi_tce_delivname'].fillna(df['koi_tce_delivname'].mode()[0])

In [16]:
for column in df.columns[df.isna().sum() > 0]:
    df[column] = df[column].fillna(df[column].mean())

In [17]:
df.isna().sum().sum()

np.int64(0)

In [18]:
df

,koi_disposition,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_period_err1,koi_period_err2,koi_time0bk,koi_time0bk_err1,koi_time0bk_err2,koi_impact,koi_impact_err1,koi_impact_err2,koi_duration,koi_duration_err1,koi_duration_err2,koi_depth,koi_depth_err1,koi_depth_err2,koi_prad,koi_prad_err1,koi_prad_err2,koi_teq,koi_insol,koi_insol_err1,koi_insol_err2,koi_model_snr,koi_tce_plnt_num,koi_tce_delivname,koi_steff,koi_steff_err1,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag
0,CONFIRMED,0,0,0,0,9.488036,0.000028,-0.000028,170.538750,0.002160,-0.002160,0.146,0.318,-0.146,2.9575,0.0819,-0.0819,615.8,19.5,-19.5,2.26,0.26,-0.15,793.0,93.59,29.45,-16.65,35.8,1.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
1,CONFIRMED,0,0,0,0,54.418383,0.000248,-0.000248,162.513840,0.003520,-0.003520,0.586,0.059,-0.443,4.5070,0.1160,-0.1160,874.8,35.5,-35.5,2.83,0.32,-0.19,443.0,9.11,2.87,-1.62,25.8,2.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
2,CONFIRMED,0,0,0,0,2.525592,0.000004,-0.000004,171.595550,0.001130,-0.001130,0.701,0.235,-0.478,1.6545,0.0420,-0.0420,603.3,16.9,-16.9,2.75,0.88,-0.35,1406.0,926.16,874.33,-314.24,40.9,1.0,q1_q17_dr25_tce,6031.0,169.0,-211.0,4.438,0.070,-0.210,1.046,0.334,-0.133,288.75488,48.226200,15.509
3,CONFIRMED,0,0,0,0,11.094321,0.000020,-0.000020,171.201160,0.001410,-0.001410,0.538,0.030,-0.428,4.5945,0.0610,-0.0610,1517.5,24.2,-24.2,3.90,1.27,-0.42,835.0,114.81,112.85,-36.70,66.5,1.0,q1_q17_dr25_tce,6046.0,189.0,-232.0,4.486,0.054,-0.229,0.972,0.315,-0.105,296.28613,48.224670,15.714
4,CONFIRMED,0,0,0,0,4.134435,0.000010,-0.000010,172.979370,0.001900,-0.001900,0.762,0.139,-0.532,3.1402,0.0673,-0.0673,686.0,18.7,-18.7,2.77,0.90,-0.30,1160.0,427.65,420.33,-136.70,40.2,2.0,q1_q17_dr25_tce,6046.0,189.0,-232.0,4.486,0.054,-0.229,0.972,0.315,-0.105,296.28613,48.224670,15.714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4536,CANDIDATE,0,0,0,0,4.736816,0.000147,-0.000147,131.787600,0.025600,-0.025600,0.218,0.285,-0.218,2.8400,1.0000,-1.0000,35.3,12.6,-12.6,0.60,0.20,-0.06,1137.0,395.05,377.30,-120.72,6.9,1.0,q1_q17_dr25_tce,6088.0,165.0,-201.0,4.456,0.056,-0.224,1.011,0.329,-0.110,289.20331,44.505138,13.922
4537,CANDIDATE,0,0,0,0,130.235324,0.003030,-0.003030,218.271900,0.020100,-0.020100,0.075,0.387,-0.075,5.6780,0.5340,-0.5340,750.1,91.4,-91.4,2.44,0.68,-0.23,332.0,2.86,2.38,-0.80,9.7,1.0,q1_q17_dr25_tce,5616.0,166.0,-183.0,4.529,0.036,-0.192,0.903,0.251,-0.084,289.57452,44.519939,15.991
4538,CANDIDATE,0,0,0,0,8.870416,0.000009,-0.000009,137.481093,0.000869,-0.000869,1.206,70.610,-0.033,1.2864,0.0514,-0.0514,873.1,25.8,-25.8,39.46,11.10,-16.68,1151.0,414.26,360.89,-292.07,43.8,1.0,q1_q17_dr25_tce,6022.0,200.0,-181.0,4.027,0.434,-0.186,1.514,0.426,-0.640,290.14914,50.239178,13.579
4539,CANDIDATE,0,0,0,0,47.109631,0.000194,-0.000194,144.131720,0.003430,-0.003430,1.230,6.923,-0.605,5.7410,0.1720,-0.1720,752.2,22.2,-22.2,78.98,30.94,-57.45,751.0,75.40,89.11,-70.44,35.1,1.0,q1_q17_dr25_tce,5258.0,159.0,-159.0,3.597,0.968,-0.242,2.780,1.089,-2.022,296.15601,44.920090,13.731


In [19]:
df['koi_tce_delivname'].unique()

<StringArray>
['q1_q17_dr25_tce', 'q1_q17_dr24_tce', 'q1_q16_tce']
Length: 3, dtype: str

In [20]:
# One-hot encode the koi_tce_delivname column
delivname_dummies = pd.get_dummies(df['koi_tce_delivname'], prefix='delivname', dtype=int)
df = pd.concat([df, delivname_dummies], axis=1)
df = df.drop('koi_tce_delivname', axis=1)

In [21]:
df

,koi_disposition,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_period_err1,koi_period_err2,koi_time0bk,koi_time0bk_err1,koi_time0bk_err2,koi_impact,koi_impact_err1,koi_impact_err2,koi_duration,koi_duration_err1,koi_duration_err2,koi_depth,koi_depth_err1,koi_depth_err2,koi_prad,koi_prad_err1,koi_prad_err2,koi_teq,koi_insol,koi_insol_err1,koi_insol_err2,koi_model_snr,koi_tce_plnt_num,koi_steff,koi_steff_err1,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag,delivname_q1_q16_tce,delivname_q1_q17_dr24_tce,delivname_q1_q17_dr25_tce
0,CONFIRMED,0,0,0,0,9.488036,0.000028,-0.000028,170.538750,0.002160,-0.002160,0.146,0.318,-0.146,2.9575,0.0819,-0.0819,615.8,19.5,-19.5,2.26,0.26,-0.15,793.0,93.59,29.45,-16.65,35.8,1.0,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347,0,0,1
1,CONFIRMED,0,0,0,0,54.418383,0.000248,-0.000248,162.513840,0.003520,-0.003520,0.586,0.059,-0.443,4.5070,0.1160,-0.1160,874.8,35.5,-35.5,2.83,0.32,-0.19,443.0,9.11,2.87,-1.62,25.8,2.0,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347,0,0,1
2,CONFIRMED,0,0,0,0,2.525592,0.000004,-0.000004,171.595550,0.001130,-0.001130,0.701,0.235,-0.478,1.6545,0.0420,-0.0420,603.3,16.9,-16.9,2.75,0.88,-0.35,1406.0,926.16,874.33,-314.24,40.9,1.0,6031.0,169.0,-211.0,4.438,0.070,-0.210,1.046,0.334,-0.133,288.75488,48.226200,15.509,0,0,1
3,CONFIRMED,0,0,0,0,11.094321,0.000020,-0.000020,171.201160,0.001410,-0.001410,0.538,0.030,-0.428,4.5945,0.0610,-0.0610,1517.5,24.2,-24.2,3.90,1.27,-0.42,835.0,114.81,112.85,-36.70,66.5,1.0,6046.0,189.0,-232.0,4.486,0.054,-0.229,0.972,0.315,-0.105,296.28613,48.224670,15.714,0,0,1
4,CONFIRMED,0,0,0,0,4.134435,0.000010,-0.000010,172.979370,0.001900,-0.001900,0.762,0.139,-0.532,3.1402,0.0673,-0.0673,686.0,18.7,-18.7,2.77,0.90,-0.30,1160.0,427.65,420.33,-136.70,40.2,2.0,6046.0,189.0,-232.0,4.486,0.054,-0.229,0.972,0.315,-0.105,296.28613,48.224670,15.714,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4536,CANDIDATE,0,0,0,0,4.736816,0.000147,-0.000147,131.787600,0.025600,-0.025600,0.218,0.285,-0.218,2.8400,1.0000,-1.0000,35.3,12.6,-12.6,0.60,0.20,-0.06,1137.0,395.05,377.30,-120.72,6.9,1.0,6088.0,165.0,-201.0,4.456,0.056,-0.224,1.011,0.329,-0.110,289.20331,44.505138,13.922,0,0,1
4537,CANDIDATE,0,0,0,0,130.235324,0.003030,-0.003030,218.271900,0.020100,-0.020100,0.075,0.387,-0.075,5.6780,0.5340,-0.5340,750.1,91.4,-91.4,2.44,0.68,-0.23,332.0,2.86,2.38,-0.80,9.7,1.0,5616.0,166.0,-183.0,4.529,0.036,-0.192,0.903,0.251,-0.084,289.57452,44.519939,15.991,0,0,1
4538,CANDIDATE,0,0,0,0,8.870416,0.000009,-0.000009,137.481093,0.000869,-0.000869,1.206,70.610,-0.033,1.2864,0.0514,-0.0514,873.1,25.8,-25.8,39.46,11.10,-16.68,1151.0,414.26,360.89,-292.07,43.8,1.0,6022.0,200.0,-181.0,4.027,0.434,-0.186,1.514,0.426,-0.640,290.14914,50.239178,13.579,0,0,1
4539,CANDIDATE,0,0,0,0,47.109631,0.000194,-0.000194,144.131720,0.003430,-0.003430,1.230,6.923,-0.605,5.7410,0.1720,-0.1720,752.2,22.2,-22.2,78.98,30.94,-57.45,751.0,75.40,89.11,-70.44,35.1,1.0,5258.0,159.0,-159.0,3.597,0.968,-0.242,2.780,1.089,-2.022,296.15601,44.920090,13.731,0,0,1


In [22]:
df.select_dtypes('object')

,koi_disposition
0,CONFIRMED
1,CONFIRMED
2,CONFIRMED
3,CONFIRMED
4,CONFIRMED
...,...
4536,CANDIDATE
4537,CANDIDATE
4538,CANDIDATE
4539,CANDIDATE


In [23]:
df['koi_disposition'] = df['koi_disposition'].replace({'CONFIRMED': 1, 'CANDIDATE': 0}).astype(int)

In [24]:
# Split df into X and y
y = df['koi_disposition']
X = df.drop('koi_disposition', axis=1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

In [25]:
X_train

,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_period_err1,koi_period_err2,koi_time0bk,koi_time0bk_err1,koi_time0bk_err2,koi_impact,koi_impact_err1,koi_impact_err2,koi_duration,koi_duration_err1,koi_duration_err2,koi_depth,koi_depth_err1,koi_depth_err2,koi_prad,koi_prad_err1,koi_prad_err2,koi_teq,koi_insol,koi_insol_err1,koi_insol_err2,koi_model_snr,koi_tce_plnt_num,koi_steff,koi_steff_err1,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag,delivname_q1_q16_tce,delivname_q1_q17_dr24_tce,delivname_q1_q17_dr25_tce
1673,0,0,0,0,5.422043,6.847000e-05,-6.847000e-05,133.460210,0.008640,-0.008640,0.9400,0.037,-0.6400,2.95200,0.22300,-0.22300,79.1,8.2,-8.2,1.39,0.26,-0.21,1222.0,527.11,272.08,-178.89,10.7,2.0,6123.0,122.0,-134.0,4.263,0.137,-0.112,1.263,0.232,-0.190,286.34430,37.411968,13.925,0,0,1
1239,0,0,0,0,81.315305,9.352000e-04,-9.352000e-04,175.995700,0.010500,-0.010500,0.0130,0.453,-0.0130,5.80700,0.22800,-0.22800,715.5,39.6,-39.6,2.55,0.30,-0.33,393.0,5.64,1.84,-1.61,20.7,1.0,5325.0,79.0,-79.0,4.390,0.120,-0.080,0.975,0.115,-0.126,294.56937,45.671261,15.111,0,0,1
3589,0,0,0,0,493.742420,1.035000e-02,-1.035000e-02,149.114500,0.012200,-0.012200,0.2377,0.236,-0.2376,5.68500,0.58100,-0.58100,506.1,45.2,-45.2,7.15,4.37,-1.55,358.0,3.90,7.20,-1.72,12.4,1.0,5015.0,116.0,-115.0,3.429,0.203,-0.344,3.243,1.984,-0.705,285.34329,39.413849,12.518,1,0,0
415,0,0,0,0,3.584101,1.916000e-06,-1.916000e-06,170.427829,0.000422,-0.000422,0.0450,0.319,-0.0450,2.73160,0.02060,-0.02060,2608.1,18.1,-18.1,4.66,1.34,-0.44,1131.0,386.66,332.86,-109.75,166.3,1.0,5739.0,171.0,-188.0,4.518,0.048,-0.204,0.922,0.264,-0.088,286.48227,46.692451,15.954,0,0,1
2925,0,0,0,0,114.336052,3.815000e-03,-3.815000e-03,216.796600,0.028000,-0.028000,0.5290,0.019,-0.4550,14.07700,0.81400,-0.81400,239.8,21.5,-21.5,2.09,0.35,-0.35,429.0,8.02,3.54,-2.77,13.0,1.0,5802.0,78.0,-78.0,4.218,0.162,-0.108,1.316,0.222,-0.222,290.14655,39.665821,15.015,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2895,0,0,0,0,8.917286,2.228000e-04,-2.228000e-04,139.605100,0.023400,-0.023400,0.0620,0.381,-0.0620,4.46800,0.67600,-0.67600,196.3,34.4,-34.4,1.39,0.45,-0.15,931.0,177.84,170.85,-55.41,7.2,1.0,6210.0,175.0,-219.0,4.467,0.054,-0.216,0.986,0.320,-0.107,296.89600,48.047260,14.671,0,0,1
2763,0,0,0,0,17.807566,8.324000e-04,-8.324000e-04,146.883230,0.006230,-0.006230,0.4740,0.001,-0.4730,1.64300,0.15000,-0.15000,1511.0,162.0,-162.0,3.92,1.20,-0.51,742.0,71.56,66.53,-25.40,11.8,1.0,6194.0,197.0,-241.0,4.450,0.070,-0.210,0.981,0.299,-0.128,294.42126,39.628471,15.574,0,0,1
905,0,0,0,0,30.183702,1.154000e-04,-1.154000e-04,146.646750,0.003110,-0.003110,0.0110,0.392,-0.0110,5.87900,0.10300,-0.10300,974.9,28.9,-28.9,3.65,0.57,-0.62,634.0,38.20,17.02,-13.79,37.9,2.0,5733.0,115.0,-104.0,4.281,0.143,-0.104,1.183,0.184,-0.202,294.31699,43.629341,15.505,0,0,1
3980,0,0,0,0,12.253843,8.837000e-05,-8.837000e-05,143.705490,0.005110,-0.005110,0.4410,0.529,-0.2440,2.37300,0.19800,-0.19800,147.7,14.3,-14.3,1.00,0.13,-0.03,678.0,50.05,16.42,-5.98,12.3,2.0,5413.0,73.0,-81.0,4.579,0.011,-0.105,0.820,0.100,-0.027,285.31381,48.725250,13.747,0,0,1


In [26]:
# Scale X
scaler = StandardScaler()
scaler.fit(X_train)

X_train = pd.DataFrame(scaler.transform(X_train), index=X_train.index, columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)

In [27]:
y_train.value_counts()

koi_disposition
0    1589
1    1589
Name: count, dtype: int64

### Training

In [28]:
models = {
    'Logistic Regression': LogisticRegression(),
    '      Decision Tree': DecisionTreeClassifier(),
    '     Neural Network': MLPClassifier(),
    '      Random Forest': RandomForestClassifier(),
    '           LightGBM': LGBMClassifier(),
    '           CatBoost': CatBoostClassifier(verbose=0),
    '            XGBoost': XGBClassifier()
}

In [29]:
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name + " trained.")        

Logistic Regression trained.
      Decision Tree trained.
     Neural Network trained.
      Random Forest trained.
[LightGBM] [Info] Number of positive: 1589, number of negative: 1589
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000929 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8583
[LightGBM] [Info] Number of data points in the train set: 3178, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
           LightGBM trained.
           CatBoost trained.
            XGBoost trained.


### Results

In [30]:
y_pred = model.predict(X_test)

In [32]:
def get_classification(y_test, y_pred, positive_label=1):
    tp = 0
    fp = 0
    fn = 0
    tn = 0
    for y_t, y_p in zip(y_test, y_pred):
        if y_t == positive_label:
            if y_p == positive_label:
                tp += 1
            else:
                fn += 1
        else:
            if y_p == positive_label:
                fp += 1
            else:
                tn += 1
    return tp, fn, fp, tn

In [33]:
get_classification(y_test, y_pred, positive_label=1)

(603, 101, 142, 517)

In [34]:
def get_accuracy(tp, fn, fp, tn):
    acc = (tp + tn) / (tp + fn + fp + tn)
    return acc

In [35]:
get_accuracy(*get_classification(y_test, y_pred, positive_label=1))

0.8217168011738811

In [36]:
def get_precision(tp, fn, fp, tn):
    precision = tp / (tp + fp)
    return precision

In [37]:
def get_recall(tp, fn, fp, tn):
    recall = tp / (tp + fn)
    return recall

In [38]:
def get_f1_score(tp, fn, fp, tn):
    precision = get_precision(tp, fn, fp, tn)
    recall = get_recall(tp, fn, fp, tn)
    f1_score = (2 * precision * recall) / (precision + recall)
    return f1_score

In [42]:
# Accuracy
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(name + " Accuracy: {:.3f}%".format(get_accuracy(*get_classification(y_test, y_pred))*100))

Logistic Regression Accuracy: 79.751%
      Decision Tree Accuracy: 75.275%
     Neural Network Accuracy: 80.558%
      Random Forest Accuracy: 81.585%
           LightGBM Accuracy: 81.731%
           CatBoost Accuracy: 82.392%
            XGBoost Accuracy: 82.172%


In [43]:
# F1 Score
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(name + " F1 Score: {:.5f}".format(get_f1_score(*get_classification(y_test, y_pred))))

Logistic Regression F1 Score: 0.81673
      Decision Tree F1 Score: 0.75980
     Neural Network F1 Score: 0.81507
      Random Forest F1 Score: 0.82435
           LightGBM F1 Score: 0.82910
           CatBoost F1 Score: 0.83516
            XGBoost F1 Score: 0.83230
